## 👨‍💻 Solution 1.4: User-Defined and Generic Types

Let's apply what we've learned about user-defined and generic types:

**Class type annotations:**

1. Create a `Product` class with appropriate type annotations for its attributes and methods. The class should have:
   - A name (string)
   - A price (float)
   - An optional description (string or None)
   - A method to calculate the price after discount
   - A method to create a formatted description string

**Generic classes:**

2. Create a generic `Repository` class that can store and retrieve items of any type. Implement:
   - A method to add items
   - A method to get an item by ID
   - A method to get all items
   - A method to remove an item

**Protocols:**

3. Define a `Serializable` protocol that requires objects to have:
   - A `to_dict()` method that returns a dictionary
   - A `from_dict()` class method that creates an object from a dictionary

Then implement two classes that satisfy this protocol.



*Class type annotations:*

In [ ]:
# 1. Create Product class with attributes name, price, and description

from typing import Optional

class Product:
    def __init__(self, name: str, price: float, description: Optional[str] = None):
        self.name: str = name
        self.price: float = price
        self.description: Optional[str] = description

    def calculate_discounted_price(self, discount_percent: float) -> float:
        """Calculate the price after applying a discount percentage."""
        if not 0 <= discount_percent <= 100:
            raise ValueError("Discount percentage must be between 0 and 100")
        return self.price * (1 - discount_percent / 100)

    def get_formatted_description(self, include_price: bool = True) -> str:
        """Return a formatted product description."""
        base_description = self.description or "No description available"
        if include_price:
            return f"{self.name} - ${self.price:.2f}: {base_description}"
        return f"{self.name}: {base_description}"

# Usage example
laptop: Product = Product(
    "Laptop Pro",
    1299.99,
    "A powerful laptop for developers"
)

sale_price = laptop.calculate_discounted_price(15)
print(f"Sale price: ${sale_price:.2f}")
print(laptop.get_formatted_description())

*Generic Repository class:*


In [ ]:
# 2. Generic Repository class to store items of any type

from typing import TypeVar, Generic, Dict, List, Optional, Any

T = TypeVar('T')

class Repository(Generic[T]):
    def __init__(self):
        self.items: Dict[str, T] = {}

    def add(self, item_id: str, item: T) -> None:
        """Add an item to the repository."""
        self.items[item_id] = item

    def get(self, item_id: str) -> Optional[T]:
        """Get an item by ID, or None if not found."""
        return self.items.get(item_id)

    def get_all(self) -> List[T]:
        """Get all items in the repository."""
        return list(self.items.values())

    def remove(self, item_id: str) -> bool:
        """Remove an item by ID. Returns True if item was removed, False if not found."""
        if item_id in self.items:
            del self.items[item_id]
            return True
        return False

    def count(self) -> int:
        """Return the number of items in the repository."""
        return len(self.items)

# Usage example with Product
product_repo: Repository[Product] = Repository()
product_repo.add("laptop-001", Product("Laptop Pro", 1299.99))
product_repo.add("phone-001", Product("Smartphone X", 899.99))

# Usage example with another type
user_repo: Repository[dict[str, Any]] = Repository()
user_repo.add("user-001", {"name": "Alice", "email": "alice@example.com"})
user_repo.add("user-002", {"name": "Bob", "email": "bob@example.com"})

*Serializable protocol and implementing classes:*


In [ ]:
# 3. Implementing a Serializable protocol for JSON serialization

from typing import Protocol, Dict, Any, Type, ClassVar, List
from dataclasses import dataclass
from datetime import datetime, date

class Serializable(Protocol):
    def to_dict(self) -> Dict[str, Any]:
        """Convert the object to a dictionary."""
        ...

    @classmethod
    def from_dict(cls: Type["Serializable"], data: Dict[str, Any]) -> "Serializable":
        """Create an object from a dictionary."""
        ...

# First implementing class
@dataclass
class User:
    name: str
    email: str
    birth_date: date
    active: bool = True

    def to_dict(self) -> Dict[str, Any]:
        return {
            "name": self.name,
            "email": self.email,
            "birth_date": self.birth_date.isoformat(),
            "active": self.active
        }

    @classmethod
    def from_dict(cls, data: Dict[str, Any]) -> "User":
        # Convert ISO date string to date object
        if isinstance(data["birth_date"], str):
            data["birth_date"] = date.fromisoformat(data["birth_date"])
        return cls(**data)

# Second implementing class
class Task:
    def __init__(
        self,
        title: str,
        description: str,
        due_date: Optional[datetime] = None,
        completed: bool = False,
        tags: List[str] = None
    ):
        self.title = title
        self.description = description
        self.due_date = due_date
        self.completed = completed
        self.tags = tags or []
        self.created_at = datetime.now()

    def to_dict(self) -> Dict[str, Any]:
        return {
            "title": self.title,
            "description": self.description,
            "due_date": self.due_date.isoformat() if self.due_date else None,
            "completed": self.completed,
            "tags": self.tags,
            "created_at": self.created_at.isoformat()
        }

    @classmethod
    def from_dict(cls, data: Dict[str, Any]) -> "Task":
        # Make a copy to avoid modifying the input
        task_data = data.copy()

        # Convert date strings to datetime objects
        if task_data.get("due_date"):
            task_data["due_date"] = datetime.fromisoformat(task_data["due_date"])

        # Remove created_at from the data for __init__
        created_at_str = task_data.pop("created_at", None)

        # Create the task
        task = cls(**task_data)

        # Set created_at if it was in the data
        if created_at_str:
            task.created_at = datetime.fromisoformat(created_at_str)

        return task

# Function that works with any Serializable object
def save_to_json_file(obj: Serializable, filename: str) -> None:
    """Save a serializable object to a JSON file."""
    import json
    with open(filename, 'w') as f:
        json.dump(obj.to_dict(), f, indent=2)

# Usage example
user = User("Alice Smith", "alice@example.com", date(1990, 5, 15))
task = Task("Complete project", "Finish the quarterly project", datetime(2023, 6, 30))

# Both User and Task can be used with the save_to_json_file function
# save_to_json_file(user, "user.json")
# save_to_json_file(task, "task.json")